
# ED Pathway Trainer — v13.3

Predict **final admitted unit** (ICU/StepDown vs others) from EMS+ED signals **and** operational context.
Adds: occupancy-at-arrival features, arrival hour sin/cos, scaled vitals/labs, stratified split, pos_weight.


In [62]:
# === ED Agent Mesh — minimal embed (v1.5-no-QR core) ===

import os, random, json
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List
from pathlib import Path
from datetime import datetime, timezone

# 0) Determinism + pandas safety
try:
    import numpy as np
    np.random.seed(0)
except Exception:
    pass
random.seed(0)
try:
    import pandas as pd
    pd.set_option("mode.copy_on_write", True)  # avoid chained-assignment foot-guns
except Exception:
    pass

# 1) World / policies
@dataclass
class ICUState:
    total: int = 2
    occupied: int = 2
    jokers: int = 0
    def free(self) -> int:
        return max(self.total - self.occupied, 0)

@dataclass
class CTPolicy:
    cooldown_minutes: int = 180   # block repeat CT within 3h
    last_ct_minute: Optional[int] = None

@dataclass
class BedPolicy:
    request_cooldown_minutes: int = 30  # prevent spam every <30 min
    last_request_minute: Optional[int] = None

@dataclass
class World:
    minute: int = 0
    icu: ICUState = field(default_factory=ICUState)
    ct: CTPolicy = field(default_factory=CTPolicy)
    bed: BedPolicy = field(default_factory=BedPolicy)

W = World()

# 2) Event bus (single entry point for all actions)
class EventBus:
    def __init__(self, world: World):
        self.world = world
        self.log: List[str] = []

    def _p(self, *parts):
        line = " | ".join(parts)
        print(line)
        self.log.append(line)

    def _permit(self, action, reason="All invariants satisfied"): self._p("permit", action, reason)
    def _block (self, action, reason):                         self._p("block ", action, reason)
    def _emit  (self, action, params):                         self._p("emit  ", action, str(params))
    def _action(self, action, msg="emitted"):                  self._p("action", action, msg)
    def _audit (self, action, msg="completed"):                self._p("audit ", action, msg)

    def _check(self, action: str, params: Dict[str, Any]):
        # ICU bed gating
        if action == "proposal.bed.request":
            if self.world.icu.free() <= 0:
                return False, "ICU full — propose ED hold or transfer"
            if self.world.bed.last_request_minute is not None:
                if self.world.minute - self.world.bed.last_request_minute < self.world.bed.request_cooldown_minutes:
                    remain = (self.world.bed.request_cooldown_minutes - (self.world.minute - self.world.bed.last_request_minute)) * 60
                    return False, f"cooldown {int(remain)}s"
        # CT repeat cooldown
        if action == "proposal.order.ct":
            if self.world.ct.last_ct_minute is not None:
                if self.world.minute - self.world.ct.last_ct_minute < self.world.ct.cooldown_minutes:
                    return False, "repeat not allowed"
        return True, "All invariants satisfied"

    def propose(self, action: str, params: Dict[str, Any] = None):
        if params is None: params = {}
        ok, reason = self._check(action, params)
        if ok:
            self._permit(action, reason)
            # side-effects
            if action == "proposal.bed.request":
                self.world.bed.last_request_minute = self.world.minute
            if action == "proposal.order.ct":
                self.world.ct.last_ct_minute = self.world.minute
            self._emit(action, params)
            self._action(action, "emitted")
            self._audit(action, "completed")
        else:
            self._block(action, reason)

eventbus = EventBus(W)

# 3) Clinical helper stubs (demo-level, NOT for clinical use)
def ecg_to_traffic_light(ecg: Dict[str, Any]) -> str:
    """
    Map ECG AI output to GREEN/YELLOW/RED (very rough).
    Expects: {'severity': 'NORMAL|URGENT|CRITICAL', 'metrics': {'QTc': int}}
    """
    sev = (ecg.get("severity") or "").upper()
    qtc = int(ecg.get("metrics", {}).get("QTc", 0) or 0)
    if sev == "CRITICAL": return "RED"
    if qtc >= 500:        return "YELLOW"
    if sev == "URGENT":   return "YELLOW"
    return "GREEN"

# Default assay key (flip later if your site differs)
TROP_ASSAY = os.getenv("TROP_ASSAY", "Abbott_Architect_hs_cTnI")

def troponin_hint_abbott(t0: float, t1: float, sex: str = "F") -> str:
    """
    Naive demo thresholds for hs-cTnI (Abbott). Do NOT use clinically.
    """
    delta = t1 - t0
    if t1 >= 52 or delta >= 10:      return "RED"
    if delta <= 2 and t1 < 16:       return "GREEN"
    return "YELLOW"

def trop_hint(t0: float, t1: float, sex: str = "F", assay: str = TROP_ASSAY) -> str:
    if assay == "Abbott_Architect_hs_cTnI":
        return troponin_hint_abbott(t0, t1, sex)
    # elif assay == "Roche_hs_cTnT": return troponin_hint_roche(t0, t1, sex)  # stub later
    return troponin_hint_abbott(t0, t1, sex)

# 4) Public API you can call from the trainer
def mesh_emit(kind: str, params: Dict[str, Any] | None = None):
    """
    Send an action through the mesh so gating/cooldowns apply uniformly.
    Examples:
      mesh_emit("proposal.order.ecg", {"priority": "STAT"})
      mesh_emit("proposal.order.labs", {"panel_id": "ed_big_panel", "priority": "STAT"})
      mesh_emit("proposal.request.vbga", {"site": "venous"})
      mesh_emit("proposal.order.ct", {"protocol": "head_trauma", "priority": "STAT"})
      mesh_emit("proposal.bed.request", {"service": "ICU"})
    """
    eventbus.propose(kind, params or {})

# Helpers if your trainer needs clock/capacity
def mesh_advance(minutes: int): W.minute += minutes
def mesh_set_icu(total=None, occupied=None, jokers=None):
    if total is not None: W.icu.total = total
    if occupied is not None: W.icu.occupied = occupied
    if jokers is not None: W.icu.jokers = jokers
    print(f"ICU capacity: total={W.icu.total}, occupied={W.icu.occupied}, free={W.icu.free()}")

# 5) (Optional) KPIs + audit (write to Kaggle working dir)
from collections import defaultdict
kpi = defaultdict(int)
_timers = {}
def kpi_start(key): _timers[key] = W.minute
def kpi_stop(key):
    if key in _timers: kpi[f"t_{key}"] = W.minute - _timers[key]

AUDIT = Path(os.getenv("ED_WORKDIR", "/kaggle/working")) / "ed_mesh_audit.jsonl"
def audit_event(tag, payload=None):
    AUDIT.parent.mkdir(parents=True, exist_ok=True)
    with open(AUDIT, "a", encoding="utf-8") as f:
        f.write(json.dumps({"ts": datetime.now(timezone.utc).isoformat(),
                            "min": W.minute, "tag": tag, "payload": payload or {}}) + "\n")

# 6) (Optional) quick smoke sanity — set to True to run once
RUN_MESH_SMOKE = False
if RUN_MESH_SMOKE:
    mesh_set_icu(total=2, occupied=2)
    mesh_emit("proposal.bed.request", {"service":"ICU"})  # expect block when full
    W.ct.last_ct_minute = None
    mesh_emit("proposal.order.ct", {"protocol":"head_trauma","priority":"STAT"})  # permit
    m0 = W.minute
    mesh_emit("proposal.order.ct", {"protocol":"head_trauma","priority":"STAT"})  # block repeat
    assert W.ct.last_ct_minute == m0
    assert ecg_to_traffic_light({"severity":"CRITICAL","metrics":{"QTc":460}}) == "RED"
    assert ecg_to_traffic_light({"severity":"URGENT","metrics":{"QTc":480}}) == "YELLOW"
    assert ecg_to_traffic_light({"severity":"NORMAL","metrics":{"QTc":430}}) == "GREEN"
    print("mesh smoke OK")
# ----------------------------------------------------------------------


In [63]:
# Cell 4 — write the Roche module locally and hard-import it (Option B)
from pathlib import Path
import sys, importlib

MODULE_PATH = Path.cwd() / 'flow_troponin_fixed.py'
MODULE_SRC = r'''\
"""flow_troponin_fixed.py
Canonical troponin rules for PoC/demo (NOT for clinical use).

Implements: Roche Elecsys hs-cTnT (ng/L), ESC 0/1-hour algorithm.
Returns ('rule_in'|'rule_out'|'observe', meta_dict).
"""

from typing import Tuple, Dict, Any

ASSAYS: Dict[str, Dict[str, float]] = {
    'Roche_hs_cTnT': {
        'rule_in_baseline': 52.0,
        'rule_in_delta':    5.0,
        'rule_out_baseline':12.0,
        'rule_out_delta':   3.0,
    }
}

def _is_nan(x):
    try:
        return x != x
    except Exception:
        return False

def classify_troponin_0_1h(assay_key: str, t0: float, t1: float | None) -> Tuple[str, Dict[str, Any]]:
    if assay_key not in ASSAYS:
        return 'observe', {'trigger': 'unknown_assay', 'assay': assay_key, 't0': t0, 't1': t1, 'delta': None}

    if t0 is None or _is_nan(t0) or (t1 is not None and _is_nan(t1)):
        return 'observe', {'trigger': 'invalid_value', 'assay': assay_key, 't0': t0, 't1': t1, 'delta': None}
    if t0 < 0 or (t1 is not None and t1 < 0):
        return 'observe', {'trigger': 'negative_value', 'assay': assay_key, 't0': t0, 't1': t1, 'delta': None}

    a = ASSAYS[assay_key]

    if t1 is None:
        return 'observe', {'trigger': 'needs_serial', 'assay': assay_key, 't0': t0, 't1': None, 'delta': None}

    delta = t1 - t0

    if (t0 >= a['rule_in_baseline']) or (delta >= a['rule_in_delta']):
        return 'rule_in', {'trigger': f"baseline>={a['rule_in_baseline']:.1f}_or_delta>={a['rule_in_delta']:.1f}", 'assay': assay_key, 't0': t0, 't1': t1, 'delta': delta}

    if (t0 < a['rule_out_baseline']) and (delta < a['rule_out_delta']):
        return 'rule_out', {'trigger': f"t0<{a['rule_out_baseline']:.1f}_and_delta<{a['rule_out_delta']:.1f}", 'assay': assay_key, 't0': t0, 't1': t1, 'delta': delta}

    return 'observe', {'trigger': 'observe_zone', 'assay': assay_key, 't0': t0, 't1': t1, 'delta': delta}
'''

MODULE_PATH.write_text(MODULE_SRC, encoding='utf-8')
sys.path.insert(0, str(Path.cwd()))
import flow_troponin_fixed as troponin
importlib.reload(troponin)
from flow_troponin_fixed import classify_troponin_0_1h as trop01
print('[troponin] source:', Path(troponin.__file__).resolve())

ASSAY_KEY = 'Roche_hs_cTnT'
def classify_trop(t0, t1, *, ckd_stage=None):
    label, meta = trop01(ASSAY_KEY, t0, t1)
    if label == 'rule_in' and (ckd_stage is not None and ckd_stage >= 3):
        meta = {**meta, 'ckd_guard': True}
        label = 'observe'
    return label, meta

assert classify_trop(11.9, 13.8)[0] == 'rule_out'
assert classify_trop(20.0, 25.1)[0] == 'rule_in'
assert classify_trop(52.0, 53.0)[0] == 'rule_in'
assert classify_trop(20.0, 22.0)[0] == 'observe'
print('Roche 0/1h wiring OK')

[troponin] source: /kaggle/working/flow_troponin_fixed.py
Roche 0/1h wiring OK


In [64]:
# Cell 5 — troponin flow scheduler: 1h pair + 3h fallback (mesh-wired)
from typing import Callable, Dict, Any, Optional
import heapq, itertools

for _name in ('mesh_emit','mesh_advance','W','classify_trop'):
    if _name not in globals():
        raise RuntimeError(f'{_name} is not defined. Run previous cells first.')

_EVENT_Q = []
_SEQ = itertools.count()

def plan_after(minutes: int, fn: Callable, *args, **kwargs):
    due = W.minute + int(minutes)
    heapq.heappush(_EVENT_Q, (due, next(_SEQ), fn, args, kwargs))

def run_until(target_minute: int):
    target_minute = int(target_minute)
    while _EVENT_Q and _EVENT_Q[0][0] <= target_minute:
        due, _, fn, args, kwargs = heapq.heappop(_EVENT_Q)
        if due > W.minute: mesh_advance(due - W.minute)
        fn(*args, **kwargs)
    if target_minute > W.minute:
        mesh_advance(target_minute - W.minute)

def run_all(max_steps: int = 1000):
    steps = 0
    while _EVENT_Q and steps < max_steps:
        due, _, fn, args, kwargs = heapq.heappop(_EVENT_Q)
        if due > W.minute: mesh_advance(due - W.minute)
        fn(*args, **kwargs)
        steps += 1

_TROP_DB: Dict[str, Dict[str, float]] = {}
_TROP_PROVIDER: Optional[Callable[[str, str], Optional[float]]] = None

def set_troponin_provider(fn: Callable[[str, str], Optional[float]]):
    global _TROP_PROVIDER
    _TROP_PROVIDER = fn

def record_trop(enc_id: str, timepoint: str, value: float):
    _TROP_DB.setdefault(enc_id, {})[timepoint] = float(value)

def fetch_trop(enc_id: str, timepoint: str) -> Optional[float]:
    if _TROP_PROVIDER is not None:
        v = _TROP_PROVIDER(enc_id, timepoint)
        if v is not None: return float(v)
    return _TROP_DB.get(enc_id, {}).get(timepoint)

def schedule_0h_bundle(enc_id: str, *, priority='STAT'):
    mesh_emit('proposal.order.ecg', {'priority': priority})
    mesh_emit('proposal.order.labs', {'panel_id': 'troponin_0h', 'priority': priority})
    plan_after(60, schedule_1h_bundle, enc_id, priority=priority)
    plan_after(61, _trop_pair_decision, enc_id)

def schedule_1h_bundle(enc_id: str, *, priority='STAT'):
    mesh_emit('proposal.order.ecg', {'priority': priority})
    mesh_emit('proposal.order.labs', {'panel_id': 'troponin_1h', 'priority': priority})

def _trop_pair_decision(enc_id: str, *, ckd_stage: Optional[int] = None, retry_min=5, max_retries=6):
    t0 = fetch_trop(enc_id, '0h')
    t1 = fetch_trop(enc_id, '1h')
    if t0 is None or t1 is None:
        if max_retries > 0:
            plan_after(retry_min, _trop_pair_decision, enc_id, ckd_stage=ckd_stage, retry_min=retry_min, max_retries=max_retries-1)
        else:
            print(f'TROP | {enc_id} | waiting for results (0/1h) gave up')
        return
    label, meta = classify_trop(t0, t1, ckd_stage=ckd_stage)
    trig = meta.get('trigger','')
    print(f'TROP | {enc_id} | {t0:.1f} -> {t1:.1f} ng/L | {label} | {trig}')
    if label == 'rule_in':
        mesh_emit('proposal.ecg.alert', {'severity':'CRITICAL','phenotype':'TroponinRuleIn','confidence':0.99})
        return
    if label == 'rule_out':
        return
    mesh_emit('proposal.order.labs', {'panel_id': 'troponin_3h', 'priority': 'STAT'})
    plan_after(120, _trop_0_3h_decision, enc_id, ckd_stage=ckd_stage)

def _trop_0_3h_decision(enc_id: str, *, ckd_stage: Optional[int] = None, retry_min=5, max_retries=6):
    t0 = fetch_trop(enc_id, '0h')
    t3 = fetch_trop(enc_id, '3h')
    if t0 is None or t3 is None:
        if max_retries > 0:
            plan_after(retry_min, _trop_0_3h_decision, enc_id, ckd_stage=ckd_stage, retry_min=retry_min, max_retries=max_retries-1)
        else:
            print(f'TROP | {enc_id} | waiting for results (0/3h) gave up')
        return
    delta3 = t3 - t0
    if t0 >= 52 or delta3 >= 6:
        if ckd_stage is not None and ckd_stage >= 3:
            print(f'TROP | {enc_id} | 0/3h -> OBSERVE (CKD guard) | t0={t0:.1f} d3={delta3:.1f}')
        else:
            print(f'TROP | {enc_id} | 0/3h -> RULE_IN | t0={t0:.1f} d3={delta3:.1f}')
            mesh_emit('proposal.ecg.alert', {'severity':'CRITICAL','phenotype':'TroponinRuleIn_3h','confidence':0.95})
        return
    if t0 < 12 and delta3 < 3:
        print(f'TROP | {enc_id} | 0/3h -> RULE_OUT | t0={t0:.1f} d3={delta3:.1f}')
        return
    print(f'TROP | {enc_id} | 0/3h -> OBSERVE | t0={t0:.1f} d3={delta3:.1f}')

def start_chest_pain_pathway(enc_id: str, *, priority='STAT', ckd_stage: Optional[int] = None):
    schedule_0h_bundle(enc_id, priority=priority)
    plan_after(61, _trop_pair_decision, enc_id, ckd_stage=ckd_stage)
    print(f'Chest pain pathway scheduled for {enc_id} at t={W.minute} min (1h pair + 3h fallback)')

In [65]:
# --- Troponin (Roche hs-cTnT 0/1h) wiring ---
from flow_troponin_fixed import classify_troponin_0_1h as trop01  # uses ASSAYS['Roche_hs_cTnT']

ASSAY_KEY = "Roche_hs_cTnT"  # PoC: hardcode; later make it a config/env

def classify_trop(t0, t1, *, ckd_stage=None):
    label, meta = trop01(ASSAY_KEY, t0, t1)  # returns 'rule_in' | 'rule_out' | 'observe', and metadata
    if label == "rule_in" and (ckd_stage is not None and ckd_stage >= 3):
        # CKD guard: no auto finalization; treat as observe + escalate
        meta = {**meta, "ckd_guard": True}
        label = "observe"
    return label, meta

# Sanity edges
assert classify_trop(11.9, 13.8)[0] == "rule_out"
assert classify_trop(20.0, 25.1)[0] == "rule_in"


In [66]:
def schedule_1h_pair(enc_id):
    # in your sim, either enqueue a future event or just advance time in-place:
    mesh_advance(60)  # 60 minutes later
    mesh_emit("proposal.order.ecg", {"priority":"STAT"})
    mesh_emit("proposal.order.labs", {"panel_id":"troponin_1h", "priority":"STAT"})


In [67]:

CFG = {"use_ingestion": False,
       "synth_n_patients": 300, "synth_hours": 12,
       "train": {"epochs": 8, "batch_size": 64, "lr": 1e-3, "hidden": 64, "dropout": 0.1, "early_stop_patience": 3},
       "label_rule": "icu_or_stepdown", "max_seq_len": 24,
       "val_split": 0.15, "test_split": 0.15,
       "random_seed": 20250811}

import os, math, json, random, time, zipfile
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Literal, Any, Tuple, Iterable
import numpy as np, pandas as pd
import torch, torch.nn as nn

SEED = CFG["random_seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False

def assert_no_na(df: pd.DataFrame, cols: Iterable[str]):
    cols = [c for c in cols if c in df.columns]
    if not cols: return True
    na = df[cols].isna().sum()
    if int(na.sum())>0: raise ValueError(f"NA values present: {{ {', '.join(f'{k}:{int(v)}' for k,v in na.items() if v>0)} }}")
    return True

print("[env] ready | seed:", SEED)


[env] ready | seed: 20250811


In [68]:

ActionReq = Literal["AUTO","TRIAGE_RN","ATTENDING_CONFIRM"]
@dataclass
class Hospital:
    name: str; coords: Tuple[float,float]; capabilities: set
    capacity: Dict[str,int] = field(default_factory=lambda: {"ICU":4,"StepDown":8,"EDObs":12,"Ward":60})
    occupied: Dict[str,int] = field(default_factory=dict)
    nurse_slots: Dict[str,int] = field(default_factory=lambda: {"ICU":4,"StepDown":10,"EDObs":12,"Ward":60})
    vents: int = 4; vent_in_use: int = 0
    def available(self, unit: str) -> int: return int(self.capacity.get(unit,0) - self.occupied.get(unit,0))
    def admit(self, unit: str, *, needs_vent: bool=False) -> bool:
        if self.available(unit) <= 0: return False
        if self.nurse_slots.get(unit,0) <= 0: return False
        if needs_vent and (self.vent_in_use >= self.vents): return False
        self.occupied[unit] = self.occupied.get(unit,0) + 1
        self.nurse_slots[unit] = self.nurse_slots.get(unit,0) - 1
        if needs_vent: self.vent_in_use += 1
        return True
    def occ_ratio(self, unit:str)->float:
        cap = self.capacity.get(unit,0); occ = self.occupied.get(unit,0)
        return float(occ/max(cap,1)) if cap>0 else 1.0
    def apply_icu_joker_if_allowed(self, *, attending_ok: bool) -> bool:
        # Minimal joker: add 1 ICU bed if >=90% occupied and attending approves
        cap = self.capacity.get("ICU",0); occ = self.occupied.get("ICU",0)
        ratio = (occ/max(cap,1)) if cap>0 else 1.0
        if ratio>=0.9 and attending_ok:
            self.capacity["ICU"] = self.capacity.get("ICU",0) + 1
            self.nurse_slots["ICU"] = self.nurse_slots.get("ICU",0) + 1
            return True
        return False

class BedManager:
    def __init__(self, hospital: Hospital, region: List[Hospital]):
        self.h=hospital; self.region=[r for r in region if r.name!=hospital.name]
    def request_transfer(self, *, target_unit: str, attending_ok: bool, needs_vent: bool=False):
        if not attending_ok: return {"approved": False, "reason": "attending approval required"}
        if self.h.admit(target_unit, needs_vent=needs_vent):
            return {"approved": True, "final_unit": target_unit, "to_hospital": self.h.name, "transferred": False}
        if target_unit == "ICU" and self.h.apply_icu_joker_if_allowed(attending_ok=True):
            if self.h.admit("ICU", needs_vent=needs_vent):
                return {"approved": True, "final_unit": "ICU", "to_hospital": self.h.name, "transferred": False, "joker_used": True}
        for other in self.region:
            if other.admit(target_unit, needs_vent=needs_vent):
                return {"approved": True, "final_unit": target_unit, "to_hospital": other.name, "transferred": True}
        return {"approved": True, "final_unit": "ED-BOARD", "to_hospital": None, "transferred": None, "board_in_ed": True}

class XRTechAgent:
    def __init__(self, hospital: Hospital): self.h=hospital; self.queue: List[str]=[]
    def request_transfer(self, pid: str): self.queue.append(pid)
    def tick(self):
        if self.queue: return {"pid": self.queue.pop(0), "status": "xr_done"}
        return None

class TransferCoordinatorAgent:
    def __init__(self, bed_manager: BedManager): self.bm=bed_manager; self.queue: List[Dict[str,Any]]=[]
    def request(self, pid: str, unit: str, attending_ok: bool, needs_vent: bool=False):
        self.queue.append({"pid": pid, "unit": unit, "attending_ok": attending_ok, "needs_vent": needs_vent})
    def tick(self):
        if not self.queue: return None
        req=self.queue.pop(0)
        res=self.bm.request_transfer(target_unit=req["unit"], attending_ok=req["attending_ok"], needs_vent=req["needs_vent"])
        return {"pid": req["pid"], **res}

class AttendingAgent:
    def approve_transfer(self, pid: str, unit: str)->bool: return True

print("[agents] ok")


[agents] ok


In [69]:

def _triage(sbp,hr,spo2,gcs, stemi=False, trauma=False):
    si = hr/max(sbp,1)
    if stemi or trauma: return "red"
    if (sbp<90) or (gcs<9) or (spo2<88) or (si>=1.1): return "red"
    return "yellow_green"

class BedState:
    def __init__(self, capacity:Dict[str,int], nurse_slots:Dict[str,int], vents:int):
        self.capacity=capacity.copy(); self.occ={k:0 for k in capacity}; self.nurse=nurse_slots.copy()
        self.vents=vents; self.vent_use=0
    def admit(self, unit:str, needs_vent=False):
        if self.occ[unit]>=self.capacity[unit] or self.nurse.get(unit,0)<=0: return False
        if needs_vent and self.vent_use>=self.vents: return False
        self.occ[unit]+=1; self.nurse[unit]=self.nurse.get(unit,0)-1
        if needs_vent: self.vent_use+=1
        return True
    def occ_ratio(self, unit:str)->float:
        cap=self.capacity.get(unit,0); occ=self.occ.get(unit,0); return float(occ/max(cap,1)) if cap>0 else 1.0
    def icu_joker(self):
        cap=self.capacity.get("ICU",0); occ=self.occ.get("ICU",0); ratio=1.0 if cap==0 else occ/max(cap,1)
        if ratio>=0.9:
            self.capacity["ICU"]=self.capacity.get("ICU",0)+1; self.nurse["ICU"]=self.nurse.get("ICU",0)+1; return True
        return False

def generate_synth(n=300, hours=12, start=pd.Timestamp("2025-08-10 08:00")):
    prev = {"sepsis":0.18,"acs":0.12,"major_trauma":0.06,"stroke":0.05,"pregnant_syncope":0.03,"other":0.56}
    conds = list(prev.keys()); p = np.array(list(prev.values())); p/=p.sum()
    chosen = list(np.random.choice(conds, size=n, p=p))
    sex = np.where(np.random.rand(n)<0.52, "female","male"); age = np.random.randint(18, 95, size=n)
    arrivals = [start + pd.Timedelta(minutes=float(np.random.uniform(0, hours*60))) for _ in range(n)]
    rows_pat=[]; rows_ems=[]; rows_labs=[]; rows_tx=[]

    cand=[]
    for i in range(n):
        cond=chosen[i]
        if cond=="sepsis":
            sbp,hr,spo2,gcs = int(np.clip(np.random.normal(95,18),60,150)), int(np.clip(np.random.normal(110,20),60,170)), int(np.clip(np.random.normal(93,4),80,100)), int(np.clip(np.random.normal(13.5,1.5),8,15))
        elif cond=="acs":
            sbp,hr,spo2,gcs = int(np.clip(np.random.normal(115,20),80,170)), int(np.clip(np.random.normal(95,18),50,160)), int(np.clip(np.random.normal(95,3),85,100)), int(np.clip(np.random.normal(14.5,0.8),10,15))
        elif cond=="major_trauma":
            sbp,hr,spo2,gcs = int(np.clip(np.random.normal(90,25),60,160)), int(np.clip(np.random.normal(110,25),60,180)), int(np.clip(np.random.normal(92,6),75,100)), int(np.clip(np.random.normal(12.0,3.0),3,15))
        elif cond=="stroke":
            sbp,hr,spo2,gcs = int(np.clip(np.random.normal(150,25),90,220)), int(np.clip(np.random.normal(85,15),45,150)), int(np.clip(np.random.normal(95,3),85,100)), int(np.clip(np.random.normal(13.0,2.0),6,15))
        else:
            sbp,hr,spo2,gcs = int(np.clip(np.random.normal(120,20),80,180)), int(np.clip(np.random.normal(88,18),45,160)), int(np.clip(np.random.normal(96,2),88,100)), int(np.clip(np.random.normal(14.5,1.0),10,15))
        stemi = (cond=="acs") and (np.random.rand()<0.12); trauma=(cond=="major_trauma")
        triage=_triage(sbp,hr,spo2,gcs,stemi,trauma); arr=arrivals[i]

        cand.append({"pid":f"S{i:05d}","cond":cond,"sex":sex[i],"age":int(age[i]),"arr":arr,"sbp":sbp,"hr":hr,"spo2":spo2,"gcs":gcs,
                     "stemi":stemi,"trauma":trauma,"triage":triage,
                     "lactate": float(np.round(np.random.lognormal(np.log(2.2 if cond=='sepsis' else 1.5),0.3),2)),
                     "hs-ctnt": float(np.round(np.random.lognormal(np.log(80),0.6),1)) if (cond=='acs' and np.random.rand()<0.35) else float(np.round(np.random.uniform(2.0,8.0),1))})
    cand = sorted(cand, key=lambda r: r["arr"])

    beds = BedState({"ICU":4,"StepDown":8,"EDObs":12,"Ward":60}, {"ICU":4,"StepDown":10,"EDObs":12,"Ward":60}, 4)

    for r in cand:
        pid=r["pid"]; arr=r["arr"]
        occ_icu, occ_sd, occ_ed = beds.occ_ratio("ICU"), beds.occ_ratio("StepDown"), beds.occ_ratio("EDObs")
        if r["stemi"] or r["trauma"]: req="ICU"
        elif r["triage"]=="red": req="StepDown"
        else: req="EDObs" if r["cond"] in ("sepsis","pregnant_syncope") else "Ward"
        needs_vent = (r["cond"]=="sepsis" and r["gcs"]<=12) or (r["trauma"] and r["gcs"]<9)
        accepted = beds.admit(req, needs_vent=needs_vent)
        joker_used=False; regional=False; boarded=False; final=req if accepted else None
        if not accepted:
            if req=="ICU" and beds.icu_joker(): joker_used=True; accepted=beds.admit("ICU", needs_vent=needs_vent); final="ICU" if accepted else None
            if not accepted:
                if np.random.rand()<0.25: regional=True; accepted=True; final=req
                else: boarded=True; accepted=False; final="ED-BOARD"

        rows_pat.append({"pid":pid,"sex":r["sex"],"age":r["age"],"condition":r["cond"],"arrival":arr,
                         "arr_hour_sin": math.sin(2*math.pi*arr.hour/24.0), "arr_hour_cos": math.cos(2*math.pi*arr.hour/24.0),
                         "occ_icu": occ_icu, "occ_sd": occ_sd, "occ_edobs": occ_ed})
        rows_ems.append({"pid":pid,"arrival":arr,"sbp":r["sbp"],"hr":r["hr"],"spo2":r["spo2"],"gcs":r["gcs"],"stemi":r["stemi"],"trauma":r["trauma"],"triage":r["triage"]})
        rows_labs += [
            {"pid":pid,"test":"lactate","value":r["lactate"],"result_at":arr + pd.Timedelta(minutes=50)},
            {"pid":pid,"test":"hs-ctnt","value":r["hs-ctnt"],"result_at":arr + pd.Timedelta(minutes=55)},
        ]
        rows_tx.append({"pid":pid,"requested_unit":req,"final_unit":final,"accepted":bool(accepted),
                        "regional":bool(regional),"boarded":bool(boarded)})

    patients=pd.DataFrame(rows_pat).sort_values("arrival")
    ems=pd.DataFrame(rows_ems).sort_values("arrival")
    labs=pd.DataFrame(rows_labs).sort_values(["pid","result_at"])
    transfers=pd.DataFrame(rows_tx).sort_values(["pid"])

    for df, cols in [(patients,["pid","arrival"]), (ems,["pid","arrival","triage"]), (labs,["pid","test","result_at","value"]), (transfers,["pid","final_unit"])]:
        assert_no_na(df, [c for c in cols if c in df.columns])

    return patients, ems, labs, transfers

patients, ems, labs, transfers = generate_synth(CFG["synth_n_patients"], CFG["synth_hours"])
print("[data] patients:", patients.shape, "| ems:", ems.shape, "| labs:", labs.shape, "| transfers:", transfers.shape,
      "| transfers.final_unit uniq:", sorted(transfers["final_unit"].astype(str).unique())[:10])


[data] patients: (300, 10) | ems: (300, 9) | labs: (600, 4) | transfers: (300, 6) | transfers.final_unit uniq: ['ED-BOARD', 'EDObs', 'ICU', 'StepDown', 'Ward']


In [70]:

def _scale_01(x, lo, hi): return np.clip((x - lo) / (hi - lo), 0.0, 1.0)

def build_dataset(patients, ems, labs, transfers, *, max_seq_len=24, label_rule="icu_or_stepdown"):
    P = patients[["pid","sex","age","arrival","arr_hour_sin","arr_hour_cos","occ_icu","occ_sd","occ_edobs"]].copy()
    E = ems[["pid","arrival","sbp","hr","spo2","gcs","triage"]].copy()
    L = labs[["pid","test","value","result_at"]].copy()
    T = transfers.copy()

    sex_map = {"male":0, "female":1}
    P.loc[:, "sex_f"] = P["sex"].map(sex_map).fillna(0).astype("float32")
    P.loc[:, "age_f"] = _scale_01(P["age"].astype(float), 18, 95).astype("float32")
    for col in ["arr_hour_sin","arr_hour_cos","occ_icu","occ_sd","occ_edobs"]:
        P.loc[:, col] = P[col].astype("float32")

    static_cols = ["sex_f","age_f","arr_hour_sin","arr_hour_cos","occ_icu","occ_sd","occ_edobs"]
    X_static = P[["pid"]+static_cols].copy()

    step_minutes = 30
    vitals_cols = ["sbp","hr","spo2","gcs"]
    lab_cols = ["lactate","hs-ctnt"]
    seq_data = {}

    E = E.sort_values(["pid","arrival"])
    L = L.sort_values(["pid","result_at"])

    for _, prow in P.iterrows():
        pid = prow["pid"]; t0 = pd.to_datetime(prow["arrival"])
        grid = [t0 + pd.Timedelta(minutes=step_minutes*k) for k in range(max_seq_len)]
        vs = E[E["pid"]==pid][["arrival"]+vitals_cols].values.tolist()
        lp = L[L["pid"]==pid][["result_at","test","value"]].values.tolist()
        cur = {c: np.nan for c in vitals_cols}
        seq = []
        i_v=0; i_l=0
        for t in grid:
            while i_v < len(vs) and pd.to_datetime(vs[i_v][0]) <= t:
                _, sbp, hr, spo2, gcs = vs[i_v]; cur["sbp"]=float(sbp); cur["hr"]=float(hr); cur["spo2"]=float(spo2); cur["gcs"]=float(gcs); i_v+=1
            feat = [cur[c] if (cur[c]==cur[c]) else np.nan for c in vitals_cols]
            lv = {"lactate": np.nan, "hs-ctnt": np.nan}
            while i_l < len(lp) and pd.to_datetime(lp[i_l][0]) <= t:
                _, test, val = lp[i_l]; test=str(test).lower()
                if test in lv: lv[test]=float(val)
                i_l+=1
            feat += [lv["lactate"], lv["hs-ctnt"]]
            seq.append(feat)
        arr = np.array(seq, dtype="float32")
        # forward fill + backfill
        for j in range(arr.shape[1]):
            last = np.nan
            for i in range(arr.shape[0]):
                if not (arr[i,j]==arr[i,j]): arr[i,j]=last
                else: last=arr[i,j]
            if not (arr[0,j]==arr[0,j]):
                med = np.nanmedian(arr[:,j]); arr[:,j] = np.nan_to_num(arr[:,j], nan=float(med) if np.isfinite(med) else 0.0)
        # clip then scale
        rngs = [(60,200),(40,200),(80,100),(3,15),(0.3,15.0),(1.0,300.0)]
        for j,(lo,hi) in enumerate(rngs): arr[:,j] = np.clip(arr[:,j], lo, hi)
        arr[:,0] = _scale_01(arr[:,0], 60,200)  # sbp
        arr[:,1] = _scale_01(arr[:,1], 40,200)  # hr
        arr[:,2] = _scale_01(arr[:,2], 80,100)  # spo2
        arr[:,3] = _scale_01(arr[:,3], 3,15)    # gcs
        arr[:,4] = _scale_01(arr[:,4], 0.3,15)  # lactate
        arr[:,5] = np.clip(np.log1p(arr[:,5]) / np.log1p(300.0), 0.0, 1.0)  # hs-ctnt

        seq_data[pid]=arr

    def final_unit_of(pid: str)->Optional[str]:
        row = T[T["pid"]==pid]
        if row.empty: return None
        u = row.iloc[0]
        for key in ["final_unit","admitted_unit","unit","requested_unit"]:
            if key in row.columns and pd.notna(u.get(key, None)):
                return str(u.get(key))
        return None

    def make_label(pid):
        fu = final_unit_of(pid)
        if fu is None or fu == "" or str(fu)=="None": return 0.0
        fu = str(fu).upper()
        if label_rule == "icu": return 1.0 if fu=="ICU" else 0.0
        if label_rule == "icu_or_stepdown": return 1.0 if fu in ("ICU","STEPDOWN") else 0.0
        if label_rule == "edobs_or_higher": return 1.0 if fu in ("ICU","STEPDOWN","EDOBS") else 0.0
        return 0.0

    y = {pid: make_label(pid) for pid in P["pid"]}
    return X_static, seq_data, y, static_cols, (vitals_cols+lab_cols)

X_static, seq_data, y, static_cols, ts_cols = build_dataset(patients, ems, labs, transfers, max_seq_len=CFG["max_seq_len"], label_rule=CFG["label_rule"])
print("[features] static:", len(static_cols), "| ts:", len(ts_cols), "| patients:", len(seq_data))


[features] static: 7 | ts: 6 | patients: 300


In [71]:

from torch.utils.data import Dataset, DataLoader

class PathwayDataset(Dataset):
    def __init__(self, X_static: pd.DataFrame, seq_data: Dict[str,np.ndarray], y: Dict[str,float]):
        self.pids = [pid for pid in X_static["pid"].tolist() if pid in seq_data and pid in y]
        self.static = X_static.set_index("pid").loc[self.pids].astype("float32").values
        self.seq = [seq_data[pid].astype("float32") for pid in self.pids]
        self.y = np.array([y[pid] for pid in self.pids], dtype="float32")
    def __len__(self): return len(self.pids)
    def __getitem__(self, idx):
        return torch.from_numpy(self.seq[idx]), torch.from_numpy(self.static[idx]), torch.tensor(self.y[idx])

ds = PathwayDataset(X_static, seq_data, y)
y_all = np.array([y[pid] for pid in ds.pids], dtype=np.int32)
pos_idx = np.where(y_all==1)[0].tolist(); neg_idx = np.where(y_all==0)[0].tolist()
rng = np.random.default_rng(SEED)
rng.shuffle(pos_idx); rng.shuffle(neg_idx)

def take_split(idxs, frac): n = int(round(len(idxs)*frac)); return idxs[:n], idxs[n:]
val_pos, pos_rem = take_split(pos_idx, CFG["val_split"])
val_neg, neg_rem = take_split(neg_idx, CFG["val_split"])
test_pos, pos_rem = take_split(pos_rem, CFG["test_split"]/(1.0-CFG["val_split"]))
test_neg, neg_rem = take_split(neg_rem, CFG["test_split"]/(1.0-CFG["val_split"]))
train_idx = pos_rem + neg_rem
val_idx = val_pos + val_neg
test_idx = test_pos + test_neg
rng.shuffle(train_idx); rng.shuffle(val_idx); rng.shuffle(test_idx)

class SubsetDS(Dataset):
    def __init__(self, base: PathwayDataset, indices: List[int]):
        self.base=base; self.indices=indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]

train_ds = SubsetDS(ds, train_idx)
val_ds   = SubsetDS(ds, val_idx)
test_ds  = SubsetDS(ds, test_idx)

def make_loader(d:Dataset, bs:int, shuffle:bool): return DataLoader(d, batch_size=bs, shuffle=shuffle)

train_loader = make_loader(train_ds, CFG["train"]["batch_size"], True)
val_loader   = make_loader(val_ds,   CFG["train"]["batch_size"], False)
test_loader  = make_loader(test_ds,  CFG["train"]["batch_size"], False)

pos = float(np.sum([ds[i][2].item() for i in train_idx])); neg = float(len(train_idx)-pos)
pos_weight = torch.tensor([neg/max(pos,1.0)], dtype=torch.float32)
print(f"[split] train/val/test = {len(train_idx)}/{len(val_idx)}/{len(test_idx)} | pos_rate_train={pos/len(train_idx):.3f} | pos_weight={pos_weight.item():.2f}")

class GRUMLP(nn.Module):
    def __init__(self, ts_in:int, static_in:int, hidden:int=64, dropout:float=0.1):
        super().__init__()
        self.gru = nn.GRU(ts_in, hidden, batch_first=True)
        self.static_mlp = nn.Sequential(nn.Linear(static_in, hidden), nn.ReLU(), nn.Dropout(dropout),
                                        nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x_ts, x_static):
        _, h = self.gru(x_ts); h = h[-1]
        s = self.static_mlp(x_static)
        return self.head(torch.cat([h,s], dim=1)).squeeze(1)

model = GRUMLP(ts_in=len(ts_cols), static_in=len(static_cols), hidden=CFG["train"]["hidden"], dropout=CFG["train"]["dropout"])
opt = torch.optim.Adam(model.parameters(), lr=CFG["train"]["lr"])
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("[model] params:", sum(p.numel() for p in model.parameters()))

def roc_auc_score_np(y_true: np.ndarray, y_score: np.ndarray)->float:
    order = np.argsort(-y_score)
    y_true = y_true[order]; y_score = y_score[order]
    P = y_true.sum(); N = len(y_true) - P
    if P==0 or N==0: return float("nan")
    tp=fp=0.0; prev=None; area=last_fp=last_tp=0.0
    for yt,ys in zip(y_true,y_score):
        if prev is not None and ys!=prev: area += (fp-last_fp)*(tp+last_tp)/2.0; last_fp, last_tp = fp, tp
        prev=ys; tp += (yt==1); fp += (yt==0)
    area += (fp-last_fp)*(tp+last_tp)/2.0
    return float(area/(P*N))

def eval_loader(m, loader):
    m.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x_ts, x_stat, y in loader:
            logits = m(x_ts.float(), x_stat.float()); prob = torch.sigmoid(logits)
            ys.append(y.numpy()); ps.append(prob.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    auc = roc_auc_score_np(y, p)
    loss = float(nn.BCELoss()(torch.tensor(p), torch.tensor(y)).item())
    return {"auc": float(auc), "bce": float(loss), "n": int(len(y))}

best = {"auc": -1, "state": None}; patience = CFG["train"]["early_stop_patience"]
for epoch in range(CFG["train"]["epochs"]):
    model.train(); total=0.0
    for x_ts, x_stat, y in train_loader:
        opt.zero_grad(); logits = model(x_ts.float(), x_stat.float())
        loss = crit(logits, y.float()); loss.backward(); opt.step()
        total += float(loss.item())
    valm = eval_loader(model, val_loader)
    print(f"epoch {epoch+1}: train_loss={total/len(train_loader):.4f} | val_auc={valm['auc']:.3f} val_bce={valm['bce']:.3f}")
    if valm["auc"]>best["auc"]:
        best={"auc": valm["auc"], "state": {k:v.cpu().clone() for k,v in model.state_dict().items()}}
        patience = CFG["train"]["early_stop_patience"]
    else:
        patience -= 1
        if patience<=0:
            print("[early stop]"); break

if best["state"] is not None: model.load_state_dict(best["state"])
testm = eval_loader(model, test_loader)
print("[test]", testm)


[split] train/val/test = 210/45/45 | pos_rate_train=0.148 | pos_weight=5.77
[model] params: 26817
epoch 1: train_loss=1.2860 | val_auc=0.695 val_bce=0.709
epoch 2: train_loss=1.1857 | val_auc=0.741 val_bce=0.709
epoch 3: train_loss=1.2141 | val_auc=0.748 val_bce=0.704
epoch 4: train_loss=1.2094 | val_auc=0.774 val_bce=0.705
epoch 5: train_loss=1.1792 | val_auc=0.786 val_bce=0.697
epoch 6: train_loss=1.2484 | val_auc=0.797 val_bce=0.687
epoch 7: train_loss=1.1340 | val_auc=0.805 val_bce=0.678
epoch 8: train_loss=1.1272 | val_auc=0.816 val_bce=0.668
[test] {'auc': 0.5639097744360902, 'bce': 0.6972793936729431, 'n': 45}


In [72]:

home = Hospital("Campus A",(0,0),{"ED","PCI"})
peer = Hospital("Campus B",(2,1),{"ED","PCI"})
bm = BedManager(home, [home, peer])
att = AttendingAgent(); tx = TransferCoordinatorAgent(bm)

def disposition_from_prob(prob: float)->str:
    if prob>=0.75: return "ICU"
    if prob>=0.55: return "StepDown"
    if prob>=0.35: return "EDObs"
    return "Ward"

events=[]
model.eval()
with torch.no_grad():
    for idx in test_idx:
        pid = ds.pids[idx]
        x_ts, x_stat, ytrue = ds[idx]
        prob = float(torch.sigmoid(model(x_ts[None].float(), x_stat[None].float())).item())
        unit = disposition_from_prob(prob)
        ok = att.approve_transfer(pid, unit=unit)
        tx.request(pid, unit, attending_ok=ok, needs_vent=(unit=="ICU"))
        res = tx.tick() or {}
        events.append({"pid": pid, "prob": prob, "unit_req": unit,
                       "approved": bool(res.get("approved", False)),
                       "final_unit": res.get("final_unit"),
                       "to_hospital": res.get("to_hospital"),
                       "transferred": bool(res.get("transferred", False)),
                       "board_in_ed": bool(res.get("board_in_ed", False)),
                       "joker_used": bool(res.get("joker_used", False))})
import pandas as pd
df_events = pd.DataFrame(events)
for col, default in {"approved": False, "transferred": False, "board_in_ed": False, "joker_used": False}.items():
    if col not in df_events.columns: df_events[col] = default
    else: df_events.loc[:, col] = df_events[col].fillna(default)

metrics = {"n": int(len(df_events)),
           "icu_req_rate": float((df_events["unit_req"]=="ICU").mean()),
           "admit_rate": float(df_events["approved"].astype(bool).mean()),
           "board_rate": float(df_events["board_in_ed"].astype(bool).mean()),
           "regional_rate": float(df_events["transferred"].astype(bool).mean()),
           "final_units": dict(df_events["final_unit"].value_counts().head(10))}
df_events.head(), metrics


(      pid      prob unit_req  approved final_unit to_hospital  transferred  \
 0  S00100  0.480972    EDObs      True      EDObs    Campus A        False   
 1  S00053  0.491624    EDObs      True      EDObs    Campus A        False   
 2  S00182  0.489940    EDObs      True      EDObs    Campus A        False   
 3  S00044  0.542428    EDObs      True      EDObs    Campus A        False   
 4  S00184  0.489909    EDObs      True      EDObs    Campus A        False   
 
    board_in_ed  joker_used  
 0        False       False  
 1        False       False  
 2        False       False  
 3        False       False  
 4        False       False  ,
 {'n': 45,
  'icu_req_rate': 0.0,
  'admit_rate': 1.0,
  'board_rate': 0.4444444444444444,
  'regional_rate': 0.26666666666666666,
  'final_units': {'EDObs': 24, 'ED-BOARD': 20, 'StepDown': 1}})

In [73]:

# Save artifacts
RUN_DIR = f"/mnt/data/ed_trainer_v13_3_run_{time.strftime('%Y%m%d-%H%M%S')}"
os.makedirs(RUN_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(RUN_DIR, "model.pt"))
with open(os.path.join(RUN_DIR, "config.json"), "w") as f: json.dump(CFG, f, indent=2)
df_events.to_csv(os.path.join(RUN_DIR, "test_events.csv"), index=False)
patients.to_csv(os.path.join(RUN_DIR, "patients.csv"), index=False)
ems.to_csv(os.path.join(RUN_DIR, "ems.csv"), index=False)
labs.to_csv(os.path.join(RUN_DIR, "labs.csv"), index=False)
transfers.to_csv(os.path.join(RUN_DIR, "transfers.csv"), index=False)
with zipfile.ZipFile(os.path.join(RUN_DIR, "artifacts.zip"),"w",zipfile.ZIP_DEFLATED) as zf:
    for fn in ["model.pt","config.json","test_events.csv","patients.csv","ems.csv","labs.csv","transfers.csv"]:
        zf.write(os.path.join(RUN_DIR, fn), arcname=fn)
print("Artifacts:", RUN_DIR)


Artifacts: /mnt/data/ed_trainer_v13_3_run_20250811-161445
